In [ ]:
# ViT-g @768 for "Lost in the Museum" -- the untried end of the resolution curve
#
# Resolution was the strongest lever in this task after model capacity:
# 224 -> 392 -> 518 took ViT-L from 0.708 to 0.775 to 0.792, and ViT-g at 518
# scores 0.8285. The curve was never extended past 518 because a full pass costs
# more GPU than was available; at 768 it is about 2.8 hours, which fits in a
# fresh 30-hour weekly quota with room to spare.
#
# DINOv2 handles arbitrary input sizes by interpolating its position embeddings,
# so no retraining is involved -- 768/14 = 54.9, and 770 divides exactly into
# 55x55 patches, three times the 1369 patches seen at 518.
#
# Checkpoints every few minutes: three multi-hour jobs were lost before that was
# added, and a run this long will not be repeated.
import glob, os, time
import numpy as np, torch
from pathlib import Path
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

Image.MAX_IMAGE_PIXELS = None
DATA = Path('/kaggle/input/competitions/lost-in-the-museum-f1/archive/kaggle_dataset/kaggle_dataset')
SIZE = 770          # 55 x 55 patches at patch size 14
BATCH = 2
CKPT = '/kaggle/working/g770_ckpt.npy'
DONE = '/kaggle/working/g770_done.npy'
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

if not DATA.exists():
    hits = [d for d in glob.glob('/kaggle/input/**/', recursive=True)
            if glob.glob(os.path.join(d, '*.png'))]
    assert hits, 'no PNG directory found under /kaggle/input'
    DATA = Path(max(hits, key=lambda h: len(glob.glob(os.path.join(h, '*.png')))))
    print('fell back to', DATA)
paths = sorted(DATA.glob('*.png'))
assert len(paths) == 20000, f'expected 20000 images, got {len(paths)}'
print(len(paths), 'images', torch.cuda.get_device_name(0),
      f'{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB')


In [ ]:
# torch.hub DOWNLOADS the checkpoint. If Internet is off this raises, so fall
# back to weights attached as a Dataset -- finding that out 20 minutes into a
# booked GPU window is how a window gets lost.
try:
    model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14', verbose=False)
except Exception as e:
    print('hub load failed:', e)
    w = sorted(glob.glob('/kaggle/input/**/*vitg14*.pth', recursive=True))
    assert w, 'attach dinov2_vitg14 weights as a Dataset'
    print('loading weights from', w[0])
    model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14',
                           pretrained=False, source='local', verbose=False)
    model.load_state_dict(torch.load(w[0], map_location='cpu'))
model = model.eval().cuda().half()

class Imgs(Dataset):
    def __init__(self, paths):
        self.paths = paths
        self.tf = transforms.Compose([
            transforms.Resize((SIZE, SIZE),
                              interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            return self.tf(Image.open(self.paths[i]).convert('RGB')), i
        except Exception as e:
            print('!', self.paths[i].name, e, flush=True)
            return torch.zeros(3, SIZE, SIZE), i

def gem(pt, p=3.0, eps=1e-6):
    return pt.clamp(min=eps).pow(p).mean(1).pow(1 / p)


In [ ]:
# Resume. /kaggle/working does NOT survive into a new session, so a partial run
# is only recoverable if its output was saved and RE-ATTACHED as a Dataset.
# Check there too, otherwise a 2.8 h job that times out restarts from zero.
def _load(local, pattern, default):
    if os.path.exists(local):
        print('resuming from', local)
        return np.load(local)
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
    if hits:
        print('resuming from attached', hits[0])
        return np.load(hits[0])
    return default

feats = _load(CKPT, 'g770_ckpt.npy', np.zeros((len(paths), 3072), np.float32))
done = _load(DONE, 'g770_done.npy', np.zeros(len(paths), bool))
assert len(feats) == len(paths) and len(done) == len(paths), 'checkpoint size mismatch'
print(f'resuming with {done.sum()}/{len(paths)} already embedded')

todo = np.flatnonzero(~done)
dl = DataLoader(Imgs([paths[i] for i in todo]), batch_size=BATCH,
                num_workers=4, pin_memory=True)

t0, n, last = time.time(), 0, time.time()
with torch.no_grad():
    for x, sl in dl:
        f = model.forward_features(x.cuda(non_blocking=True).half())
        v = torch.cat([f['x_norm_clstoken'], gem(f['x_norm_patchtokens'])], 1)
        idx = todo[sl.numpy()]
        feats[idx] = v.float().cpu().numpy()
        done[idx] = True
        n += len(idx)
        if time.time() - last > 180:
            np.save(CKPT, feats); np.save(DONE, done); last = time.time()
            r = n / (time.time() - t0)
            print(f'  {done.sum()}/{len(paths)}  {r:.2f} img/s  '
                  f'ETA {(len(paths)-done.sum())/r/60:.0f} min', flush=True)

np.save(CKPT, feats); np.save(DONE, done)
np.save('/kaggle/working/features_g770.npy', feats)
print(f'complete: {done.sum()}/{len(paths)} in {(time.time()-t0)/60:.1f} min')
